# Site-specific analysis

Use the data from ver 250810.

## Package Loading

In [ ]:
install.packages("pacman")

In [ ]:
pacman::p_load(
  rio, 
  here, 
  skimr, 
  tidyverse, 
  gtsummary, 
  rstatix, 
  janitor, 
  scales, 
  flextable,
  showtext,
  ggrepel,
  patchwork
  )

In [ ]:
# Optional: Japanese font setting
font_add_google("Noto Sans JP", "noto")
showtext_auto()

## Data Preparation

In [ ]:
setwd("/home/jupyter/")
df_all <- import("all_250810.csv")

head(df_all)

In [ ]:
nrow(df_all)

In [ ]:
df_all <- df_all[-c(1, 2), ]
head(df_all)

In [ ]:
colnames(df_all) <- as.character(unlist(df_all[1, ]))
df_all <- df_all[-1, ]
rownames(df_all) <- NULL

head(df_all)
nrow(df_all)

In [ ]:
cut_row <- which(grepl("kmm", df_all$id_baseline_all))[1]
df_clean <- df_all[1:(cut_row-1), ]

tail(df_clean)

In [ ]:
dim(df_clean)

In [ ]:
colnames(df_clean)

In [ ]:
names(df_clean)[is.na(names(df_clean))] <- "X1"

In [ ]:
table(df_clean$site, useNA="ifany")

In [ ]:
table(df_clean$area, useNA="ifany")

In [ ]:
df_clean <- df_clean %>%
  mutate(
    sitec = case_when(
      site == 1                         ~ 1,
      site == 2                         ~ 2,
      site == 3                         ~ 3,
      site == 4 & area == 1             ~ 4,
      site == 4 & area == 4             ~ 5,
      site == 5                         ~ 6,
      site == 6                         ~ 7,
      site == 7                         ~ 8,
      site == 8                         ~ 9,
      site == 9                         ~ 10,
      site == 10                        ~ 11,
      site == 11                        ~ 12,
      site == 12                        ~ 13,
      site == 13                        ~ 14,
      TRUE                              ~ NA_real_
    )
  )

In [ ]:
table(df_clean$site, df_clean$sitec, useNA="ifany")

In [ ]:
head(df_clean)

## Data integration

### 1. Geographic information

In [ ]:
df_adi_wi <- import("adi_wi_250810.csv")

head(df_adi_wi)

In [ ]:
nrow(df_adi_wi)

In [ ]:
df_adi_wi <- df_adi_wi[-c(1, 2), ]

head(df_adi_wi)

In [ ]:
colnames(df_adi_wi) <- as.character(unlist(df_adi_wi[1, ]))
df_adi_wi <- df_adi_wi[-1, ]
rownames(df_adi_wi) <- NULL

head(df_adi_wi)

In [ ]:
cut_row <- which(grepl("kmm", df_adi_wi$id_baseline_all))[1]
df_adi_wi_clean <- df_adi_wi[1:(cut_row-1), ]

tail(df_adi_wi_clean)

In [ ]:
dim(df_adi_wi_clean)

In [ ]:
colnames(df_adi_wi_clean)

In [ ]:
df_check <- df_clean |>
  transmute(
    id_baseline_all,
    id_gwas_clean = as.character(id_gwas)
  ) |>
  left_join(
    df_adi_wi_clean |>
      transmute(
        id_baseline_all,
        id_gwas_adi = as.character(id_gwas)
      ) %>%
      distinct(id_baseline_all, .keep_all = TRUE),
    by = "id_baseline_all"
  )

df_check |>
  summarise(
    n_total = n(),
    n_both_available = sum(!is.na(id_gwas_clean) & !is.na(id_gwas_adi)),
    n_mismatch = sum(
      !is.na(id_gwas_clean) &
        !is.na(id_gwas_adi) &
        id_gwas_clean != id_gwas_adi
    )
  )

In [ ]:
df_adi_wi_clean_unique <- df_adi_wi_clean |>
  select(-id_gwas, -id_japonica, -id_asa, -site) |>
  distinct(id_baseline_all, .keep_all = TRUE)

df_clean_m <- df_clean |>
  left_join(df_adi_wi_clean_unique, by = "id_baseline_all")

head(df_clean_m)

In [ ]:
df_clean2 <- df_clean_m |>
  filter(!is.na(id_gwas) & id_gwas != "")

head(df_clean2)

In [ ]:
dim(df_clean2)

In [ ]:
names(df_clean2)

### 2. PRS information

In [ ]:
df_prs <- import("ssd/PRS_result/PGS002379_total_PRS.csv")
head(df_prs)

In [ ]:
nrow(df_prs)

In [ ]:
df_merged <- df_clean2 |>
  inner_join(df_prs, by = c("id_gwas" = "IID"))
head(df_merged)

In [ ]:
nrow(df_merged)

In [ ]:
sum(df_merged$id_gwas %in% df_prs$IID)

In [ ]:
sum(!df_merged$id_gwas %in% df_prs$IID)

In [ ]:
colnames(df_merged)

### 3. Site name

In [ ]:
site_table <- read.csv("site_table2.csv")
site_table$site_num <- as.character(site_table$site_num)
head(site_table)

In [ ]:
df_merged$sitec <- as.character(df_merged$sitec)
class(df_merged$site)

In [ ]:
df_merged <- df_merged %>%
  left_join(site_table, by = c("sitec" = "site_num"))

df_merged <- df_merged %>%
  mutate(site_name = factor(site_name, levels = site_table$site_name))

In [ ]:
head(df_merged)

## Definition of Type 2 Diabetes

In [ ]:
df_merged <- df_merged |>
  mutate(
    PRS_quintile = ntile(PRS_z, 5),
    
    pdm = factor(
      pdm,
      levels = c(1, 2, 3),
      labels = c("なし", "あり", "以前あり")
    )
  )

table(df_merged$PRS_quintile, df_merged$pdm)

In [ ]:
plot_df <- df_merged %>%
  mutate(
    dm_bin = ifelse(
      pdm %in% c("あり", "以前あり"),
      1,
      0
    )
  ) %>%
  group_by(PRS_quintile) %>%
  summarise(
    dm_rate = mean(dm_bin, na.rm = TRUE) * 100
  )

plot_df

In [ ]:
ggplot(plot_df,
       aes(x = factor(PRS_quintile),
           y = dm_rate)) +
  geom_col() +
  labs(
    x = "PRS quintile",
    y = "Diabetes prevalence (%)"
  )

ggsave("Diabetes prevalence_by_PRS quintile.png", 
       path = "/home/jupyter/walkabilityと糖尿病",
       width = 6,
       height = 4,
       dpi = 300)

In [ ]:
df_merged <- df_merged %>%
  mutate(
    dm_bin = ifelse(
      pdm %in% c("あり", "以前あり"),
      1,
      0
    )
  )

table(df_merged$dm_bin)

#### Logistic regression of PRS between Diabetes mellitus

In [ ]:
fit <- glm(dm_bin ~ PRS_z,
           data = df_merged,
           family = binomial())

summary(fit)

PRS associated with Diabetes.

## Check the prevalence of Diabetes by site

In [ ]:
site_dm <- df_merged %>%
  group_by(site_name) %>%
  summarise(
    n = n(),
    prevalence = mean(dm_bin == 1)
  )

site_dm

In [ ]:
mean(site_dm$prevalence)

### chi-squared test

In [ ]:
tab <- table(df_merged$dm_bin, df_merged$sitec)
chisq.test(tab)

p_site <- chisq.test(tab)$p.value
p_label <- ifelse(
  p_site < 0.001,
  "χ²検定: P < 0.001",
  paste0("χ²検定: P = ", signif(p_site, 3))
)

p_label

### Bar graph of Diabetes prevalence

In [ ]:
library(showtext)

font_add_google("Noto Sans JP", "noto")

font_families()

In [ ]:
library(dplyr)

site_labels <- tibble(
  site_name = c(
    "Chiba",
    "Aichi cancer center",
    "Okazaki",
    "Shizuoka",
    "Daikou",
    "Takashima",
    "Kyoto",
    "Rengou Fukuoka",
    "Saga",
    "Kagoshima",
    "Tokushima",
    "KOPS",
    "Shizuoka,Sakuragaoka"
  ),
  site_jp = c(
    "千葉",
    "愛知がんセンター",
    "岡崎",
    "静岡",
    "大幸",
    "高島",
    "京都",
    "連合福岡",
    "佐賀",
    "鹿児島",
    "徳島",
    "KOPS",
    "静岡・桜ヶ丘"
  )
)

In [ ]:
site_dm <- site_dm %>%
left_join(site_labels, by = "site_name")

In [ ]:
site_dm <- site_dm %>%
  mutate(
    site_jp = factor(
      site_jp,
      levels = site_labels$site_jp
    )
  )

In [ ]:
library(ggplot2)
library(showtext)

font_add_google("Noto Sans JP", "noto")
showtext_auto()
showtext_opts(dpi = 300)

ggplot(site_dm, aes(x = site_jp, y = prevalence)) +
  geom_col() +
  geom_text(
    aes(label = scales::percent(prevalence, accuracy = 0.1)),
    vjust = -0.4,
    size = 3.2
  ) +
  annotate(
    "text",
    x = 10,
    y = 0.078,
    label = p_label,
    hjust = 0,
    size = 4
  ) +
  scale_y_continuous(
    limits = c(0, 0.08),
    labels = scales::percent
  ) +
  labs(
    x = NULL,
    y = "糖尿病有病率 (%)"
  ) +
  theme_bw(base_size = 16) +
  theme(text = element_text(family = "noto"),
        aspect.ratio = 0.8,
        plot.title = element_text(size = 15,hjust = 0.5),
        axis.text.x = element_text(size = 12, angle = 45, hjust = 1),
        axis.text.y = element_text(size = 12),
        axis.title = element_text(size = 16)) 

ggsave(
  "site-specific_dmprevalence.png",
　path = "/home/jupyter/walkabilityと糖尿病",
  width = 6,
  height = 6,
  dpi = 300
)

## Add Walkability Index(JPWI) variables

In [ ]:
df_merged <- df_merged |>
  mutate(
    jpwi = parse_number(as.character(jpwi))
  )

class(df_merged$jpwi)

In [ ]:
df_merged |>
  group_by(site_name) |>
  summarise(
    n = sum(!is.na(jpwi)),
    mean_jpwi = mean(jpwi, na.rm = TRUE),
    sd_jpwi = sd(jpwi, na.rm = TRUE),
    median_jpwi = median(jpwi, na.rm = TRUE),
    min_jpwi = min(jpwi, na.rm = TRUE),
    max_jpwi = max(jpwi, na.rm = TRUE),
    .groups = "drop"
  )

### Histogram of jpwi

In [ ]:
ggplot(df_merged, aes(x = jpwi)) +
  geom_histogram(bins = 50) +
  labs(
    x = "Walkability Index",
    y = "Count",
    title = "Distribution of Walkability Index"
  ) +
  theme_classic()

ggsave("histgram_jpwi.png", 
       path = "/home/jupyter/walkabilityと糖尿病",
       width = 6,
       height = 4,
       dpi = 300)

### Box plot of jpwi by site

In [ ]:
df_merged |>
  mutate(site_num = as.numeric(site),
         site = factor(site_num)) |>
  ggplot(aes(x = site_name, y = jpwi, fill = site)) +
  geom_boxplot() +
  scale_fill_brewer(palette = "Set3") +
  labs(
    x = "Site",
    y = "Walkability index"
  ) +
  theme_bw()+
  theme(axis.text.x = element_text(angle = 45, hjust = 1)) +
  theme(legend.position = "none")

ggsave("bocplot_jpwi.png", 
       path = "/home/jupyter/walkabilityと糖尿病",
       width = 6,
       height = 4,
       dpi = 300)

### Standarized JPWI

In [ ]:
df_merged$jpwi_z <- as.numeric(scale(df_merged$jpwi))

summary(df_merged$jpwi_z)
sd(df_merged$jpwi_z, na.rm = TRUE)

## Add physical activity variables

### Definition of physical activity(MET-hours/week)

In [ ]:
vars <- c('physicalwork', 'walking', 'standing', 'sitting', 
          'exer_freq_light', 'exert_time_light', 'exer_freq_mod', 'exer_time_mod', 'exer_freq_heavy', 'exert_time_heavy')

df_merged <- df_merged %>%
  mutate(across(all_of(vars), ~ ifelse(. %in% c(0, 9), NA, .)))

vars <- c("exer_freq_light","exert_time_light",
          "exer_freq_mod","exer_time_mod",
          "exer_freq_heavy","exert_time_heavy")

counts_list <- lapply(df_merged[vars], function(x) {
  sort(table(x, useNA = "always"), decreasing = TRUE)
})

counts_list

In [ ]:
value_map_freq <- c(
  "1" = 0,
  "2" = 0.5,
  "3" = 1.5,
  "4" = 3.5,
  "5" = 6
)

columns_to_map_freq <- c(
  "exer_freq_light",
  "exer_freq_mod",
  "exer_freq_heavy"
)

df_merged <- df_merged |>
mutate(
    across(
        all_of(columns_to_map_freq),
        ~ value_map_freq[as.character(.)],
        .names = "{.col}r"
    )
)

table(df_merged$exer_freq_light, df_merged$exer_freq_lightr, useNA = "ifany")
table(df_merged$exer_freq_mod, df_merged$exer_freq_modr, useNA = "ifany")
table(df_merged$exer_freq_heavy, df_merged$exer_freq_heavyr, useNA = "ifany")

In [ ]:
value_map_time <- c(
  "1" = 0.25,
  "2" = 0.75,
  "3" = 1.5,
  "4" = 2.5,
  "5" = 3.5,
  "6" = 4.0
)

columns_to_map_time <- c(
  "exert_time_light",
  "exer_time_mod",
  "exert_time_heavy"
)

df_merged <- df_merged |>
mutate(
    across(
        all_of(columns_to_map_time),
        ~ value_map_time[as.character(.)],
        .names = "{.col}r"
    )
)

table(df_merged$exert_time_light, df_merged$exert_time_lightr, useNA = "ifany")
table(df_merged$exer_time_mod, df_merged$exer_time_modr, useNA = "ifany")
table(df_merged$exert_time_heavy, df_merged$exert_time_heavyr, useNA = "ifany")

In [ ]:
df_merged <- df_merged %>%
  mutate(
    weak = 3.4 * exer_freq_lightr * exert_time_lightr,
    weak = if_else(exer_freq_light == 1, 0, weak),
    
    moderate = 7 * exer_freq_modr * exer_time_modr,
    moderate = if_else(exer_freq_mod == 1, 0, moderate),

    intense = 10 * exer_freq_heavyr * exert_time_heavyr,
    intense = if_else(exer_freq_heavy == 1, 0, intense)
  )

df_merged <- df_merged %>%
mutate(
    methw = weak + moderate +intense
    )

class(df_merged$methw)
summary(df_merged$methw)
sum(!is.na(df_merged$methw)) 

In [ ]:
sum(df_merged$methw == 0, na.rm = TRUE) 

mean(df_merged$methw == 0, na.rm = TRUE) * 100

### Box plot and violin plot of physical activity

In [ ]:
df_merged %>%
  group_by(site_name) %>%
  summarise(
    n = n(),
    mean = mean(methw, na.rm = TRUE),
    sd = sd(methw, na.rm = TRUE),
    median = median(methw, na.rm = TRUE),
    Q1 = quantile(methw, 0.25, na.rm = TRUE),
    Q3 = quantile(methw, 0.75, na.rm = TRUE),
    min = min(methw, na.rm = TRUE),
    max = max(methw, na.rm = TRUE),
    zero_n = sum(methw == 0, na.rm = TRUE),
    zero_pct = mean(methw == 0, na.rm = TRUE) * 100
  )

In [ ]:
ggplot(df_merged, aes(x = methw)) +
  geom_histogram(bins = 50) +
  labs(
    x = "METs/week",
    y = "Count",
    title = "Distribution of METs/week"
  ) +
  theme_minimal()
ggsave("histgram_methw.png", 
       path = "/home/jupyter/walkabilityと糖尿病",
       width = 6,
       height = 4,
       dpi = 300)

df_merged$log_methw <- log1p(df_merged$methw)

ggplot(df_merged, aes(x = log_methw)) +
  geom_histogram(bins = 50) +
  labs(
    x = "log1p(methw)",
    y = "Count",
    title = "Distribution of log1p(methw)"
  ) +
  theme_minimal()
ggsave("histgram_logmethw.png", 
       path = "/home/jupyter/walkabilityと糖尿病",
       width = 6,
       height = 4,
       dpi = 300)

ggplot(df_merged, aes(x = site_name, y = methw)) +
  geom_violin(fill = "grey90", color = "grey40", trim = FALSE) +
  geom_jitter(
    width = 0.15,
    alpha = 0.15,
    size = 0.8
  ) +
  labs(
    x = "Site",
    y = "MET-hours/week",
    title = "Distribution of physical activity by site"
  ) +
  theme_classic() +
  theme(
    axis.text.x = element_text(angle = 45, hjust = 1)
  )
ggsave("violin_methw.png", 
       path = "/home/jupyter/walkabilityと糖尿病",
       width = 6,
       height = 4,
       dpi = 300)

### Log-transform physical activity

In [ ]:
df_merged$log_methw <- log1p(df_merged$methw)

summary(df_merged$log_methw)

In [ ]:
table(df_merged$site, is.na(df_merged$methw))

## Creation of covariates

### drinking status

In [ ]:
table(df_merged$dr_yn, useNA = "ifany")

In [ ]:
df_merged <- df_merged %>%
  mutate(
    alcohol = case_when(
      dr_yn %in% c(1, 2, 3) ~ dr_yn,
      dr_yn %in% c(0, 9) ~ NA_character_
    )
  )

table(df_merged$alcohol, useNA = "ifany")

### smoking status

In [ ]:
table(df_merged$sm_yn, useNA = "ifany")

In [ ]:
df_merged <- df_merged %>%
  mutate(smoking = case_when(
         sm_yn %in% c(1, 2, 3) ~ sm_yn,
         sm_yn %in% c(0, 9) ~ NA_character_
    )
  )

table(df_merged$smoking, useNA = "ifany")

### Age

In [ ]:
df_merged$age <- as.numeric(df_merged$age)
class(df_merged$age)

### BMI

In [ ]:
df_merged$bmi <- as.numeric(df_merged$bmi)
class(df_merged$bmi)

df_merged$bmi[df_merged$bmi == 99999] <- NA

summary(df_merged$bmi)

## Creation of complete-Case Dataset

In [ ]:
df_complete <- df_merged %>%
  filter(
    complete.cases(
      PRS_z,
      sitec,
      age,
      sex,
      bmi,
      alcohol,
      smoking,
      jpwi,    
      jpwi_z,
      methw,
      log_methw
    )
  )

nrow(df_complete)

## Scatter plot of PRS residual and Diabetes residual

### Residuals of PRS and Diabetes (adjusted for age, sex, smoking, alcohol)

In [ ]:
fit_dm <- glm(
  dm_bin ~ age + sex + smoking + alcohol,
  family = binomial(),
  data = df_complete
)

fit_PRS <- lm(
  PRS_z ~ age + sex + smoking + alcohol,
  data = df_complete
)

df_complete$dm_resid <- residuals(
  fit_dm,
  type = "response"
)

df_complete$PRS_resid <- residuals(
  fit_PRS
)

In [ ]:
site_summary_adj <- df_complete %>%
  group_by(site_name) %>%
  summarise(
    n = n(),
    dm_resid_mean = mean(dm_resid),
    dm_resid_per = dm_resid_mean * 100,
    PRS_resid_mean = mean(PRS_resid),
    se_PRS_resid = sd(PRS_resid)/sqrt(n)
  )

site_summary_adj

In [ ]:
site_summary_adj <- site_summary_adj %>%
 left_join(site_labels, by = "site_name")

site_summary_adj

### Scatter plot

In [ ]:
cor_res_PRS <- cor.test(
  site_summary_adj$PRS_resid_mean,
  site_summary_adj$dm_resid_mean,
  method = "spearman"
)

cor_res_PRS

rho_PRS <- cor_res_PRS$estimate
pval_PRS <- cor_res_PRS$p.value

In [ ]:
ggplot(site_summary_adj,
       aes(x = PRS_resid_mean,
           y = dm_resid_per)) +
  geom_point(size = 3) +
  geom_text_repel(
  aes(label = site_jp),
  size = 4,
  segment.color = "grey50",
  segment.alpha = 0.7,
  box.padding = 0.8,
  point.padding = 0.5,
  force = 2,
  max.overlaps = Inf
) + 
  theme_bw(base_size = 14) +
  theme(plot.title = element_text(hjust = 0.5)) +
labs(
x = "平均PRS残差",
y = "糖尿病有病率残差(％)"
)

ggsave(
  "PRS_dm_resid_correlation.png",
  path = "/home/jupyter/walkabilityと糖尿病",
  width = 6,
  height = 6,
  dpi = 300
)

# Site-specific analysis

In [ ]:
site_overview <- df_complete %>%
  group_by(site_name) %>%
  summarise(
    n = n(),
    dm_n = sum(dm_bin == 1, na.rm = TRUE),
    dm_prev = mean(dm_bin, na.rm = TRUE),

    age_mean = mean(age, na.rm = TRUE),

    male_prop = mean(sex == 1, na.rm = TRUE),

    prs_mean = mean(PRS_z, na.rm = TRUE),
    prs_sd = sd(PRS_z, na.rm = TRUE),

    jpwi_mean = mean(jpwi_z, na.rm = TRUE),
    jpwi_sd = sd(jpwi_z, na.rm = TRUE),

    methw_mean = mean(log_methw, na.rm = TRUE),
    methw_sd = sd(log_methw, na.rm = TRUE),

    smoking_1_pct = mean(smoking == 1, na.rm = TRUE) * 100,
    smoking_2_pct = mean(smoking == 2, na.rm = TRUE) * 100,
    smoking_3_pct = mean(smoking == 3, na.rm = TRUE) * 100,
    
    alcohol_1_pct = mean(alcohol == 1, na.rm = TRUE) * 100,
    alcohol_2_pct = mean(alcohol == 2, na.rm = TRUE) * 100,
    alcohol_3_pct = mean(alcohol == 3, na.rm = TRUE) * 100,
    bmi_mean = mean(bmi, na.rm = TRUE),
    bmi_sd = sd(bmi, na.rm = TRUE)
  )

site_overview

In [ ]:
site_list <- split(df_complete, df_complete$sitec)

names(site_list)

In [ ]:
site_list <- split(df_complete, df_complete$site_name)

site_list <- site_list[names(site_list) != "Iga"]

### Diabetes and PRS

In [ ]:
for(site in names(site_list)){

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  fit1 <- glm(
    dm_bin ~ PRS_z + age + sex + smoking + alcohol + bmi,
    family = binomial(),
    data = dat
  )

  print(summary(fit1))
}

In [ ]:
prs_results <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit1 <- glm(
    dm_bin ~ PRS_z + age + sex + smoking + alcohol + bmi,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit1)$coefficients

  prs_results <- rbind(
    prs_results,
    data.frame(
      site = site,
      beta = coef_tab["PRS_z", "Estimate"],
      SE = coef_tab["PRS_z", "Std. Error"],
      z = coef_tab["PRS_z", "z value"],
      p = coef_tab["PRS_z", "Pr(>|z|)"]
    )
  )
}

prs_results$FDR_p <- p.adjust(
  prs_results$p,
  method = "BH"
)

prs_results <- prs_results[
  order(prs_results$FDR_p),
]

print(prs_results)

### Diabetes and JPWI

In [ ]:
for(site in names(site_list)){

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  fit2 <- glm(
    dm_bin ~ jpwi_z + age + sex + smoking + alcohol + PRS_z,
    family = binomial(),
    data = dat
  )

  print(summary(fit2))
}


In [ ]:
jpwi_results <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit2 <- glm(
    dm_bin ~ jpwi_z + age + sex + smoking + alcohol + PRS_z,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit2)$coefficients

  jpwi_results <- rbind(
    jpwi_results,
    data.frame(
      site = site,
      beta = coef_tab["jpwi_z", "Estimate"],
      SE = coef_tab["jpwi_z", "Std. Error"],
      z = coef_tab["jpwi_z", "z value"],
      p = coef_tab["jpwi_z", "Pr(>|z|)"]
    )
  )
}

jpwi_results$FDR_p <- p.adjust(
  jpwi_results$p,
  method = "BH"
)

jpwi_results <- jpwi_results[
  order(jpwi_results$FDR_p),
]

print(jpwi_results)

### Diabetes and physical activity

In [ ]:
for(site in names(site_list)){

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  fit3 <- glm(
    dm_bin ~ log_methw + age + sex + smoking + alcohol + PRS_z,
    family = binomial(),
    data = dat
  )

  print(summary(fit3))
}

In [ ]:
methw_results <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit3 <- glm(
    dm_bin ~ log_methw + age + sex + smoking + alcohol + PRS_z,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit3)$coefficients

  methw_results <- rbind(
    methw_results,
    data.frame(
      site = site,
      beta = coef_tab["log_methw", "Estimate"],
      SE = coef_tab["log_methw", "Std. Error"],
      z = coef_tab["log_methw", "z value"],
      p = coef_tab["log_methw", "Pr(>|z|)"]
    )
  )
}

methw_results$FDR_p <- p.adjust(
  methw_results$p,
  method = "BH"
)

methw_results <- methw_results[
  order(methw_results$FDR_p),
]

print(methw_results)

## Interaction analysis

### Interaction between PRS and JPWI

In [ ]:
for(site in names(site_list)){

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  fit4 <- glm(
    dm_bin ~ jpwi_z * PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  print(summary(fit4))
}

In [ ]:
jpwi_prs_int <- data.frame()

for (site in names(site_list)) {
  dat <- site_list[[site]]

  fit4 <- glm(
    dm_bin ~ jpwi_z * PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit4)$coefficients

  jpwi_prs_int <- rbind(
    jpwi_prs_int,
    data.frame(
      site = site,
      beta = coef_tab["jpwi_z", "Estimate"],
      SE   = coef_tab["jpwi_z", "Std. Error"],
      z    = coef_tab["jpwi_z", "z value"],
      p    = coef_tab["jpwi_z", "Pr(>|z|)"]
    )
    )
}

jpwi_prs_int$p_FDR <- p.adjust(jpwi_prs_int$p, method = "BH")

jpwi_prs_int <- jpwi_prs_int[
  order(jpwi_prs_int$p_FDR),
]
jpwi_prs_int

In [ ]:
jpwi_prs_int <- data.frame()

for (site in names(site_list)) {
  dat <- site_list[[site]]

  fit4 <- glm(
    dm_bin ~ jpwi_z * PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit4)$coefficients

  jpwi_prs_int <- rbind(
    jpwi_prs_int,
    data.frame(
      site = site,
      beta = coef_tab["jpwi_z:PRS_z", "Estimate"],
      SE   = coef_tab["jpwi_z:PRS_z", "Std. Error"],
      z    = coef_tab["jpwi_z:PRS_z", "z value"],
      p    = coef_tab["jpwi_z:PRS_z", "Pr(>|z|)"]
    )
    )
}

jpwi_prs_int$p_FDR <- p.adjust(jpwi_prs_int$p, method = "BH")

jpwi_prs_int <- jpwi_prs_int[
  order(jpwi_prs_int$p_FDR),
]
jpwi_prs_int

In [ ]:
lrt_results <- data.frame()

for (site in names(site_list)) {

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  m0 <- glm(
    dm_bin ~ jpwi_z + PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  m1 <- glm(
    dm_bin ~ jpwi_z * PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  lrt <- anova(m0, m1, test = "LRT")

  print(lrt)

  lrt_results <- rbind(
    lrt_results,
    data.frame(
      site = site,
      n = nrow(dat),
      cases = sum(dat$dm_bin == 1),
      beta_interaction = coef(m1)["jpwi_z:PRS_z"],
      OR_interaction = exp(coef(m1)["jpwi_z:PRS_z"]),
      LRT_chisq = lrt$Deviance[2],
      df = lrt$Df[2],
      p_value = lrt$`Pr(>Chi)`[2]
    )
  )
}

lrt_results$p_FDR <- p.adjust(lrt_results$p_value, method = "BH")

lrt_results

### Interaction between PRS and physical activity

In [ ]:
for(site in names(site_list)){

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  fit5 <- glm(
    dm_bin ~ log_methw * PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  print(summary(fit5))
}

In [ ]:
methw_prs_int <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit5 <- glm(
    dm_bin ~ log_methw * PRS_z +
      age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit5)$coefficients

  methw_prs_int <- rbind(
    methw_prs_int,
    data.frame(
      site = site,
      beta = coef_tab["log_methw", "Estimate"],
      SE   = coef_tab["log_methw", "Std. Error"],
      z    = coef_tab["log_methw", "z value"],
      p    = coef_tab["log_methw", "Pr(>|z|)"]
    )
  )
}

methw_prs_int$FDR_p <- p.adjust(
  methw_prs_int$p,
  method = "BH"
)

methw_prs_int <- methw_prs_int[
  order(methw_prs_int$FDR_p),
]

print(methw_prs_int)

In [ ]:
methw_prs_int <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit5 <- glm(
    dm_bin ~ log_methw * PRS_z +
      age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit5)$coefficients

  methw_prs_int <- rbind(
    methw_prs_int,
    data.frame(
      site = site,
      beta = coef_tab["log_methw:PRS_z", "Estimate"],
      SE   = coef_tab["log_methw:PRS_z", "Std. Error"],
      z    = coef_tab["log_methw:PRS_z", "z value"],
      p    = coef_tab["log_methw:PRS_z", "Pr(>|z|)"]
    )
  )
}

methw_prs_int$FDR_p <- p.adjust(
  methw_prs_int$p,
  method = "BH"
)

methw_prs_int <- methw_prs_int[
  order(methw_prs_int$FDR_p),
]

print(methw_prs_int)

In [ ]:
lrt_results <- data.frame()

for (site in names(site_list)) {

  cat("\n\n====================\n")
  cat(site, "\n")
  cat("====================\n")

  dat <- site_list[[site]]

  m0 <- glm(
    dm_bin ~ log_methw + PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  m1 <- glm(
    dm_bin ~ log_methw * PRS_z + age + sex + smoking + alcohol,
    family = binomial(),
    data = dat
  )

  lrt <- anova(m0, m1, test = "LRT")

  print(lrt)

  lrt_results <- rbind(
    lrt_results,
    data.frame(
      site = site,
      n = nrow(dat),
      cases = sum(dat$dm_bin == 1),
      beta_interaction = coef(m1)["log_methw:PRS_z"],
      OR_interaction = exp(coef(m1)["log_methw:PRS_z"]),
      LRT_chisq = lrt$Deviance[2],
      df = lrt$Df[2],
      p_value = lrt$`Pr(>Chi)`[2]
    )
  )
}

lrt_results$p_FDR <- p.adjust(lrt_results$p_value, method = "BH")

lrt_results

## Figure creation

### Diabetes and PRS forest plot

In [ ]:
PRS_forest <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit1 <- glm(
    dm_bin ~ PRS_z + age + sex + smoking + alcohol + bmi,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit1)$coefficients

  beta <- coef_tab["PRS_z","Estimate"]
  se   <- coef_tab["PRS_z","Std. Error"]
  p    <- coef_tab["PRS_z","Pr(>|z|)"]

  PRS_forest <- rbind(
    PRS_forest,
    data.frame(
      site = site,
      beta = beta,
      SE = se,
      OR = exp(beta),
      LCL = exp(beta - 1.96 * se),
      UCL = exp(beta + 1.96 * se),
      p = p
    )
  )
}

PRS_forest <- PRS_forest %>%
  mutate(
    OR_CI = sprintf(
      "%.2f (%.2f–%.2f)",
      OR, LCL, UCL
    )
  )

PRS_forest <- PRS_forest %>%
  left_join(site_labels, by = c("site" = "site_name"))

PRS_forest <- PRS_forest %>%
  arrange(OR)

PRS_forest

PRS_forest <- PRS_forest %>%
  mutate(
    OR_CI = sprintf(
      "%.2f (%.2f–%.2f)",
      OR, LCL, UCL
    )
  )

# =========================
# left: site name
# =========================

p_site <- ggplot(
  PRS_forest,
  aes(
    x = 1,
    y = reorder(site_jp, OR)
  )
) +
  geom_text(
    aes(label = site_jp),
    hjust = 1,
    size = 5,
    family = "noto"
  ) +
  xlim(0.5, 1) +
  theme_void()

# =========================
# center： Forest
# =========================

p_forest <- ggplot(
  PRS_forest,
  aes(
    y = reorder(site_jp, OR),
    x = OR
  )
) +
  geom_vline(
    xintercept = 1,
    linetype = "dashed"
  ) +
  geom_errorbar(
    aes(
      xmin = LCL,
      xmax = UCL
    ),
    width = 0.2,
    linewidth = 0.8
  ) +
  geom_point(
    size = 3
  ) +
  coord_cartesian(
    xlim = c(
      min(PRS_forest$LCL) * 0.9,
      max(PRS_forest$UCL) * 1.05
    )
  ) +
  theme_bw(base_size = 16) +
  theme(
    text = element_text(family = "noto"),
    axis.title.y = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank()
  ) +
  labs(
    x = "オッズ比(95％信頼区間)",
    y = NULL
  )

# =========================
# right： OR (95%CI)
# =========================

p_or <- ggplot(
  PRS_forest,
  aes(
    x = 1,
    y = reorder(site_jp, OR)
  )
) +
  geom_text(
    aes(label = OR_CI),
    hjust = 0,
    size = 5,
    family = "noto"
  ) +
  xlim(1,3) +
  theme_void() +
  theme(
    text = element_text(family = "noto"),
    plot.title = element_text(
      hjust = 0,
      face = "bold",
      size = 14
    )
  )

# =========================
# integration
# =========================

p_final <-
  p_site +
  p_forest +
  p_or +
  plot_layout(
    widths = c(2.5, 4, 2.8)
  ) +
  plot_annotation(
    theme = theme(
      plot.title = element_text(
        hjust = 0.5,
        family = "noto",
        size = 18,
        face = "bold"
      )
    )
  )

p_final

ggsave(
  "site-specific_PRS_new.png",
  path = "/home/jupyter/walkabilityと糖尿病",
  plot = p_final,
  width = 8,
  height = 6,
  dpi = 300
)   

### Diabetes and JPWI forest plot

In [ ]:
jpwi_forest <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit2 <- glm(
    dm_bin ~ jpwi_z + age + sex + smoking + alcohol + PRS_z,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit2)$coefficients

  beta <- coef_tab["jpwi_z","Estimate"]
  se   <- coef_tab["jpwi_z","Std. Error"]
  p    <- coef_tab["jpwi_z","Pr(>|z|)"]

  jpwi_forest <- rbind(
    jpwi_forest,
    data.frame(
      site = site,
      beta = beta,
      SE = se,
      OR = exp(beta),
      LCL = exp(beta - 1.96 * se),
      UCL = exp(beta + 1.96 * se),
      p = p
    )
  )
}

jpwi_forest <- jpwi_forest %>%
  mutate(
    OR_CI = sprintf(
      "%.2f (%.2f–%.2f)",
      OR, LCL, UCL
    )
  )

jpwi_forest <- jpwi_forest %>%
  left_join(site_labels, by = c("site" = "site_name"))

jpwi_forest <- jpwi_forest %>%
  arrange(OR)

jpwi_forest


# =========================
# left： site name
# =========================

p_site <- ggplot(
  jpwi_forest,
  aes(
    x = 1,
    y = reorder(site_jp, OR)
  )
) +
  geom_text(
    aes(label = site_jp),
    hjust = 1,
    size = 5,
    family = "noto"
  ) +
  xlim(0.5, 1) +
  theme_void()

# =========================
# center： Forest
# =========================

p_forest <- ggplot(
  jpwi_forest,
  aes(
    y = reorder(site_jp, OR),
    x = OR
  )
) +
  geom_vline(
    xintercept = 1,
    linetype = "dashed"
  ) +
  geom_errorbar(
    aes(
      xmin = LCL,
      xmax = UCL
    ),
    width = 0.2,
    linewidth = 0.8
  ) +
  geom_point(
    size = 3
  ) +
  coord_cartesian(
    xlim = c(
      min(jpwi_forest$LCL) * 0.9,
      max(jpwi_forest$UCL) * 1.05
    )
  ) +
  theme_bw(base_size = 16) +
  theme(
    text = element_text(family = "noto"),
    axis.title.y = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank()
  ) +
  labs(
    x = "オッズ比(95％信頼区間)",
    y = NULL
  )

# =========================
# right： OR (95%CI)
# =========================

p_or <- ggplot(
  jpwi_forest,
  aes(
    x = 1,
    y = reorder(site_jp, OR)
  )
) +
  geom_text(
    aes(label = OR_CI),
    hjust = 0,
    size = 5,
    family = "noto"
  ) +
  xlim(1,3) +
  theme_void() +
  theme(
    text = element_text(family = "noto"),
    plot.title = element_text(
      hjust = 0,
      face = "bold",
      size = 14
    )
  )

# =========================
# integration
# =========================

p_final <-
  p_site +
  p_forest +
  p_or +
  plot_layout(
    widths = c(2.5, 4, 2.8)
  ) +
  plot_annotation(
    theme = theme(
      plot.title = element_text(
        hjust = 0.5,
        family = "noto",
        size = 18,
        face = "bold"
      )
    )
  )

p_final

ggsave(
  "site-specific_jpwi_new.png",
  path = "/home/jupyter/walkabilityと糖尿病",
  plot = p_final,
  width = 8,
  height = 6,
  dpi = 300
)

### Diabetes and physical activity forest plot

In [ ]:
methw_forest <- data.frame()

for(site in names(site_list)){

  dat <- site_list[[site]]

  fit3 <- glm(
    dm_bin ~ log_methw + age + sex + smoking + alcohol + PRS_z,
    family = binomial(),
    data = dat
  )

  coef_tab <- summary(fit3)$coefficients

  beta <- coef_tab["log_methw","Estimate"]
  se   <- coef_tab["log_methw","Std. Error"]
  p    <- coef_tab["log_methw","Pr(>|z|)"]

  methw_forest <- rbind(
    methw_forest,
    data.frame(
      site = site,
      beta = beta,
      SE = se,
      OR = exp(beta),
      LCL = exp(beta - 1.96 * se),
      UCL = exp(beta + 1.96 * se),
      p = p
    )
  )
}

methw_forest <- methw_forest %>%
  mutate(
    OR_CI = sprintf(
      "%.2f (%.2f–%.2f)",
      OR, LCL, UCL
    )
  )

methw_forest <- methw_forest %>%
  left_join(site_labels, by = c("site" = "site_name"))

methw_forest <- methw_forest %>%
  arrange(OR)

methw_forest

# =========================
# left： site name
# =========================

p_site <- ggplot(
  methw_forest,
  aes(
    x = 1,
    y = reorder(site_jp, OR)
  )
) +
  geom_text(
    aes(label = site_jp),
    hjust = 1,
    size = 5,
    family = "noto"
  ) +
  xlim(0.5, 1) +
  theme_void()

# =========================
# center： Forest
# =========================

p_forest <- ggplot(
  methw_forest,
  aes(
    y = reorder(site_jp, OR),
    x = OR
  )
) +
  geom_vline(
    xintercept = 1,
    linetype = "dashed"
  ) +
  geom_errorbar(
    aes(
      xmin = LCL,
      xmax = UCL
    ),
    width = 0.2,
    linewidth = 0.8
  ) +
  geom_point(
    size = 3
  ) +
  coord_cartesian(
    xlim = c(
      min(methw_forest$LCL) * 0.9,
      max(methw_forest$UCL) * 1.05
    )
  ) +
  theme_bw(base_size = 16) +
  theme(
    text = element_text(family = "noto"),
    axis.title.y = element_blank(),
    axis.text.y = element_blank(),
    axis.ticks.y = element_blank()
  ) +
  labs(
    x = "オッズ比(95％信頼区間)",
    y = NULL
  )

# =========================
# right： OR (95%CI)
# =========================

p_or <- ggplot(
  methw_forest,
  aes(
    x = 1,
    y = reorder(site_jp, OR)
  )
) +
  geom_text(
    aes(label = OR_CI),
    hjust = 0,
    size = 5,
    family = "noto"
  ) +
  xlim(1,3) +
  theme_void() +
  theme(
    text = element_text(family = "noto"),
    plot.title = element_text(
      hjust = 0,
      face = "bold",
      size = 14
    )
  )

# =========================
# integration
# =========================

p_final <-
  p_site +
  p_forest +
  p_or +
  plot_layout(
    widths = c(2.5, 4, 2.8)
  ) +
  plot_annotation(
    theme = theme(
      plot.title = element_text(
        hjust = 0.5,
        family = "noto",
        size = 18,
        face = "bold"
      )
    )
  )

p_final

ggsave(
  "site-specific_methw_new.png",
  path = "/home/jupyter/walkabilityと糖尿病",
  plot = p_final,
  width = 8,
  height = 6,
  dpi = 300
)